# 02 — MCQ Answers (Part 1)
Tính toán trực tiếp 10 câu trắc nghiệm từ dữ liệu. Mỗi cell = 1 câu.

In [2]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data_loader import load_all_tables
tables = load_all_tables()

## Q1 — Trung vị inter-order gap (khách có > 1 đơn)

In [3]:
orders = tables['orders'].sort_values(['customer_id', 'order_date'])
orders['prev_date'] = orders.groupby('customer_id')['order_date'].shift(1)
orders['gap_days'] = (orders['order_date'] - orders['prev_date']).dt.days
multi = orders[orders['prev_date'].notna()]
median_gap = multi['gap_days'].median()
print(f'Median inter-order gap: {median_gap:.1f} days')
# A=30, B=90, C=180, D=365

Median inter-order gap: 144.0 days


## Q2 — Segment có gross margin trung bình cao nhất

In [4]:
p = tables['products'].copy()
p['gross_margin'] = (p['price'] - p['cogs']) / p['price']
result = p.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(result)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


## Q3 — Lý do trả hàng phổ biến nhất với Streetwear

In [5]:
returns  = tables['returns']
products = tables['products']
streetwear = products[products['category'] == 'Streetwear'][['product_id']]
merged = returns.merge(streetwear, on='product_id')
print(merged['return_reason'].value_counts())

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


## Q4 — Traffic source có bounce_rate trung bình thấp nhất

In [6]:
wt = tables['web_traffic']
result = wt.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(result)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


## Q5 — % dòng order_items có promo_id không null

In [8]:
oi = tables['order_items']
pct = oi['promo_id'].notna().mean() * 100
print(f'Promo applied: {pct:.1f}%')

Promo applied: 38.7%


## Q6 — Age group có số đơn trung bình/khách cao nhất

In [9]:
customers = tables['customers'].dropna(subset=['age_group'])
orders    = tables['orders']
order_counts = orders.groupby('customer_id').size().reset_index(name='n_orders')
merged = customers.merge(order_counts, on='customer_id', how='left').fillna({'n_orders': 0})
result = merged.groupby('age_group')['n_orders'].mean().sort_values(ascending=False)
print(result)

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: n_orders, dtype: float64


## Q7 — Region có tổng doanh thu cao nhất
> Lưu ý: sales.csv là daily aggregate, không có region. Cần join orders -> geography.

In [10]:
orders    = tables['orders']
order_items = tables['order_items']
geography = tables['geography']
# Revenue per order = sum(unit_price * quantity)
order_rev = order_items.groupby('order_id').apply(
    lambda x: (x['unit_price'] * x['quantity']).sum()
).reset_index(name='order_revenue')
merged = orders.merge(order_rev, on='order_id').merge(geography[['zip','region']], on='zip')
result = merged.groupby('region')['order_revenue'].sum().sort_values(ascending=False)
print(result)

region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: order_revenue, dtype: float64


## Q8 — Payment method phổ biến nhất trong cancelled orders

In [11]:
orders = tables['orders']
cancelled = orders[orders['order_status'] == 'cancelled']
print(cancelled['payment_method'].value_counts())

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


## Q9 — Size có tỷ lệ trả hàng cao nhất (S/M/L/XL)

In [12]:
returns   = tables['returns']
order_items = tables['order_items']
products  = tables['products']

oi_size = order_items.merge(products[['product_id','size']], on='product_id')
oi_size = oi_size[oi_size['size'].isin(['S','M','L','XL'])]
orders_per_size = oi_size.groupby('size').size()

ret_size = returns.merge(products[['product_id','size']], on='product_id')
ret_size = ret_size[ret_size['size'].isin(['S','M','L','XL'])]
returns_per_size = ret_size.groupby('size').size()

rate = (returns_per_size / orders_per_size).sort_values(ascending=False)
print(rate)

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64


## Q10 — Installment plan có payment_value trung bình cao nhất

In [13]:
payments = tables['payments']
result = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(result)

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


## Summary — Final Answers

In [6]:
# Computed from data — verified 2026-04-19
answers = {
    'Q1':  'C',  # median gap = 144 days → closest to 180 days
    'Q2':  'D',  # Standard: gross margin 0.313 (highest)
    'Q3':  'B',  # wrong_size: 7,626 returns in Streetwear
    'Q4':  'C',  # email_campaign: avg bounce_rate 0.00446 (lowest)
    'Q5':  'C',  # 38.66% ≈ 39%
    'Q6':  'A',  # 55+: avg 5.407 orders/customer (highest)
    'Q7':  'C',  # East: 7.64B total revenue (highest)
    'Q8':  'A',  # credit_card: 28,452 cancelled orders
    'Q9':  'A',  # S: return rate 0.0565 (highest)
    'Q10': 'C',  # 6 installments: avg payment 24,447 (highest)
}
print("=== FINAL MCQ ANSWERS ===")
for q, a in answers.items():
    print(f"  {q}: {a}")

=== FINAL MCQ ANSWERS ===
  Q1: C
  Q2: D
  Q3: B
  Q4: C
  Q5: C
  Q6: A
  Q7: C
  Q8: A
  Q9: A
  Q10: C
